# Silver Layer — Clean & Risk-Flag Financial Data
Validate transactions, deduplicate accounts, derive credit-utilisation bands and basic risk indicators.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, to_date, date_format,
    hour, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Pass-through customers — clean dimension
df_cust = spark.read.format('delta').table('bronze_customers')
df_cust = df_cust.withColumn('silver_timestamp', current_timestamp())
df_cust.write.mode('overwrite').format('delta').saveAsTable('silver_customers')
print(f'Silver customers: {df_cust.count()} rows')

In [ ]:
# Clean accounts
df_acct = spark.read.format('delta').table('bronze_accounts')
w = Window.partitionBy('account_id').orderBy(col('ingestion_timestamp').desc())
df_acct = (
    df_acct.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('balance', col('balance').cast('double'))
    .withColumn('credit_limit', col('credit_limit').cast('double'))
    .withColumn('credit_utilisation_pct', col('credit_utilisation_pct').cast('double'))
    .withColumn('open_date', to_date('open_date'))
    .filter(col('balance') >= 0)
    .withColumn('utilisation_band',
        when(col('credit_limit') == 0, 'N/A')
        .when(col('credit_utilisation_pct') <= 30, 'Low')
        .when(col('credit_utilisation_pct') <= 60, 'Medium')
        .when(col('credit_utilisation_pct') <= 90, 'High')
        .otherwise('Very High'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_acct.write.mode('overwrite').format('delta').saveAsTable('silver_accounts')
print(f'Silver accounts: {df_acct.count()} rows')

In [ ]:
# Clean transactions + derive NON-LEAKY indicators (no fraud-derived columns)
df_txn = spark.read.format('delta').table('bronze_transactions')
w2 = Window.partitionBy('transaction_id').orderBy(col('ingestion_timestamp').desc())
df_txn = (
    df_txn.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('transaction_date', to_timestamp('transaction_date'))
    .withColumn('amount', col('amount').cast('double'))
    .withColumn('is_flagged_fraud', col('is_flagged_fraud').cast('boolean'))
    .filter(col('transaction_date').isNotNull())
    .filter(col('amount') > 0)
    .withColumn('transaction_date_only', date_format('transaction_date', 'yyyy-MM-dd'))
    .withColumn('transaction_hour', hour('transaction_date'))
    .withColumn('is_night_transaction', (hour('transaction_date') >= 22) | (hour('transaction_date') < 6))
    .withColumn('is_international', col('country') != 'UK')
    .withColumn('is_high_value', col('amount') > 5000)
    .withColumn('silver_timestamp', current_timestamp())
)
df_txn.write.mode('overwrite').format('delta').saveAsTable('silver_transactions')
print(f'Silver transactions: {df_txn.count()} rows')